# CardioIA - Fase 2 | Parte 1
## Extração de sintomas e sugestão de diagnóstico

Neste notebook a gente lê as frases de pacientes do arquivo `frases_sintomas.txt`, procura os sintomas que estão no mapa de conhecimento (`mapa_conhecimento.csv`) e sugere o diagnóstico mais provável para cada relato.

Usamos NLP simbólico, baseado em regras: normalização, estemização (RSLP), expressões regulares e um dicionário de sintomas ligado às doenças. Escolhemos essa abordagem porque, em saúde, é importante conseguir explicar por que uma sugestão foi feita: o resultado mostra exatamente quais sintomas levaram a cada doença.

A pontuação é simples: cada sintoma encontrado soma um ponto para as doenças ligadas a ele. Quando os dois sintomas de uma mesma linha do mapa aparecem juntos, a doença ganha um ponto extra, porque a combinação é mais forte do que um sintoma isolado.

In [37]:
#%pip install nltk

import re
import unicodedata
import pandas as pd
import nltk
from nltk.stem import RSLPStemmer
nltk.download("rslp", quiet=True)

True

### Normalização do texto
Pacientes escrevem de jeitos diferentes (com acento, sem acento, maiúscula...). Antes de comparar, deixamos tudo em minúsculo e sem acentos.

In [38]:
def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto).lower())
    texto = "".join(ch for ch in texto if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", texto).strip()

normalizar("Dor no PEITO e aperto no Tórax")

'dor no peito e aperto no torax'

### Estemização (RSLP)
Pacientes escrevem "tonta" ou "tonto", "palpitação" ou "palpitações", "incharam" ou "inchadas". O estemizador RSLP, feito para o português, reduz cada palavra ao seu radical (`tont`, `palpit`, `inch`). Assim comparamos radicais em vez do texto exato, e a busca passa a ser por palavra inteira. O RSLP precisa dos acentos para aplicar as regras, por isso estemizamos antes de normalizar.

In [39]:
stemmer = RSLPStemmer()

def radicais(texto):
    palavras = re.findall(r"\w+", str(texto).lower())
    return [normalizar(stemmer.stem(p)) for p in palavras]

radicais("Fico tonta e sinto uma palpitação; as pernas incharam")

['fic', 'tont', 'e', 'sint', 'uma', 'palpit', 'as', 'pern', 'inch']

### Leitura dos arquivos

In [40]:
with open("frases_sintomas.txt", encoding="utf-8") as arquivo:
    frases = [linha.strip() for linha in arquivo if linha.strip()]

print(f"{len(frases)} frases carregadas\n")
for i, frase in enumerate(frases, start=1):
    print(f"{i:>2}. {frase}")

10 frases carregadas

 1. Há dois dias estou com uma dor no peito em aperto que vai para o braço esquerdo e piora quando faço esforço físico, hoje não consegui nem subir a escada de casa.
 2. Sinto cansaço constante há uma semana, mesmo depois de descansar, minhas pernas ficam inchadas no fim do dia e preciso dormir com dois travesseiros para conseguir respirar.
 3. Desde ontem sinto o coração disparado e umas palpitações do nada, fiquei tonto no trabalho e tive que sentar até passar.
 4. Há três semanas sinto uma dor no peito quando caminho rápido, mas ela passa quando paro para descansar, por isso parei de ir a pé para o mercado.
 5. Faz uns dez dias que acordo com dor de cabeça na nuca e a visão fica embaçada, não consigo me concentrar no serviço e meu rosto fica vermelho.
 6. Hoje de manhã meu braço direito ficou dormente, a boca ficou torta e comecei a falar enrolado, minha esposa precisou me ajudar a levantar da cama.
 7. Há quatro dias estou com uma dor no peito que piora quando

In [41]:
mapa = pd.read_csv("mapa_conhecimento.csv")
print(f"{len(mapa)} associações no mapa de conhecimento")
mapa.head(10)

64 associações no mapa de conhecimento


,sintoma_1,sintoma_2,doenca_associada
0,dor no peito,aperto no peito,Infarto Agudo do Miocárdio
1,dor no peito em aperto,braço esquerdo,Infarto Agudo do Miocárdio
2,aperto no tórax,suor frio,Infarto Agudo do Miocárdio
3,pressão no peito,dor no braço,Infarto Agudo do Miocárdio
4,dor que irradia,mandíbula,Infarto Agudo do Miocárdio
5,dor forte no peito,náusea,Infarto Agudo do Miocárdio
6,queimação no peito,suor frio,Infarto Agudo do Miocárdio
7,dor no peito,piora quando faço esforço,Infarto Agudo do Miocárdio
8,dor no peito,esforço físico,Infarto Agudo do Miocárdio
9,dor no peito,falta de ar,Infarto Agudo do Miocárdio


In [42]:
mapa["doenca_associada"].value_counts()

doenca_associada
Infarto Agudo do Miocárdio                     10
Angina                                          7
Insuficiência Cardíaca                          7
Arritmia                                        6
Acidente Vascular Cerebral (AVC)                6
Hipertensão Arterial                            5
Trombose Venosa Profunda / Embolia Pulmonar     5
Endocardite Infecciosa                          5
Pericardite                                     4
Miocardite                                      4
Doença Arterial Periférica                      3
Estenose Aórtica                                2
Name: count, dtype: int64

### Montando a base de conhecimento
O mapa funciona como uma ontologia simples: sintomas e doenças ligados por uma relação de associação. Para cada expressão do mapa guardamos seus radicais, o texto original (para mostrar no resultado) e as doenças ligadas a ela. Também guardamos os pares (sintoma 1 + sintoma 2) de cada linha, que valem um bônus quando aparecem juntos.

In [43]:
termos = {}
pares = []

for _, linha in mapa.iterrows():
    r1 = tuple(radicais(linha["sintoma_1"]))
    r2 = tuple(radicais(linha["sintoma_2"]))
    for r, texto in ((r1, linha["sintoma_1"]), (r2, linha["sintoma_2"])):
        termos.setdefault(r, {"texto": texto, "doencas": set()})["doencas"].add(linha["doenca_associada"])
    pares.append((r1, r2, linha["doenca_associada"]))

print(f"{len(termos)} expressões distintas | {len(pares)} pares")
print(termos[tuple(radicais("dor no peito"))])
print(termos[tuple(radicais("palpitações"))])

111 expressões distintas | 64 pares
{'texto': 'dor no peito', 'doencas': {'Miocardite', 'Infarto Agudo do Miocárdio', 'Pericardite', 'Angina'}}
{'texto': 'palpitações', 'doencas': {'Arritmia'}}


### Extraindo quando o sintoma começou
Além dos sintomas, as frases trazem a informação de tempo ("há dois dias", "desde ontem"...). Uma regex simples já resolve boa parte dos casos.

In [44]:
padroes_tempo = [
    r"\b(?:ha|faz)\s+(?:uns |umas |cerca de |mais de )?\w+\s+(?:horas?|dias?|semanas?|mes|meses|anos?)\b",
    r"\bdesde ontem\b",
    r"\bdesde \w+\b",
    r"\bhoje de manha\b",
    r"\bhoje\b",
    r"\bontem\b",
]

def extrair_tempo(frase):
    texto = normalizar(frase)
    for padrao in padroes_tempo:
        achou = re.search(padrao, texto)
        if achou:
            return achou.group(0)
    return "não informado"

extrair_tempo("Faz uns dez dias que acordo com dor de cabeça")

'faz uns dez dias'

### Função de diagnóstico
1. **Busca por palavra inteira:** a frase vira uma lista de radicais e procuramos cada expressão do mapa como sequência de radicais.
2. **Negação:** se "não", "nunca" ou "nem" aparece até 3 palavras antes do sintoma, ele é descartado ("não sinto dor no peito" não conta dor no peito).
3. **Sem contagem dobrada:** um trecho contido em outro maior da mesma doença não pontua de novo ("febre" dentro de "febre que vai e volta").
4. **Pontuação:** cada sintoma soma 1 ponto para as doenças ligadas a ele, e os dois sintomas da mesma linha do mapa juntos valem 1 ponto extra.

In [45]:
NEGACAO = {"nao", "nunca", "nem"}

def encontrar_sintomas(tokens):
    achados = []
    for termo in termos:
        n = len(termo)
        for i in range(len(tokens) - n + 1):
            if tuple(tokens[i:i + n]) == termo:
                negado = any(t in NEGACAO for t in tokens[max(0, i - 3):i])
                if not negado:
                    achados.append((i, i + n, termo))
    return achados

def remover_contidos(achados):
    return {t for i, f, t in achados
            if not any(i2 <= i and f <= f2 and (f2 - i2) > (f - i) for i2, f2, _ in achados)}

def diagnosticar(frase):
    achados = encontrar_sintomas(radicais(frase))
    pontos = {}
    for doenca in {d for _, _, t in achados for d in termos[t]["doencas"]}:
        da_doenca = remover_contidos([a for a in achados if doenca in termos[a[2]]["doencas"]])
        bonus = sum(1 for r1, r2, d in pares if d == doenca and r1 in da_doenca and r2 in da_doenca)
        pontos[doenca] = len(da_doenca) + bonus
    ranking = sorted(pontos.items(), key=lambda item: item[1], reverse=True)
    sintomas_frase = sorted({termos[t]["texto"] for _, _, t in achados})
    return ranking, sintomas_frase

print(diagnosticar("Não sinto dor no peito, só dor de cabeça na nuca e a visão embaçada"))
print(diagnosticar("Fico tonta e sinto uma palpitação estranha há três dias"))

([('Hipertensão Arterial', 2)], ['dor de cabeça', 'dor de cabeça na nuca', 'nuca', 'visão embaçada'])
([('Arritmia', 3)], ['palpitações', 'tonto'])


### Rodando para os 10 pacientes

In [46]:
resultados = []

for i, frase in enumerate(frases, start=1):
    ranking, sintomas = diagnosticar(frase)
    tempo = extrair_tempo(frase)

    print(f"Paciente {i}")
    print(f"  Relato: {frase}")
    print(f"  Início dos sintomas: {tempo}")
    print(f"  Sintomas identificados: {', '.join(sintomas) if sintomas else 'nenhum'}")

    if ranking:
        principal, pontuacao = ranking[0]
        outras = [f"{d} ({p})" for d, p in ranking[1:3]]
        print(f"  Diagnóstico sugerido: {principal} (pontuação {pontuacao})")
        if outras:
            print(f"  Outras possibilidades: {', '.join(outras)}")
    else:
        principal, pontuacao, outras = "Sem sugestão", 0, []
        print("  Diagnóstico sugerido: nenhum sintoma do mapa foi encontrado")
    print()

    resultados.append({
        "paciente": i,
        "frase": frase,
        "inicio_sintomas": tempo,
        "sintomas_identificados": "; ".join(sintomas),
        "diagnostico_sugerido": principal,
        "pontuacao": pontuacao,
        "outras_possibilidades": "; ".join(outras),
    })

Paciente 1
  Relato: Há dois dias estou com uma dor no peito em aperto que vai para o braço esquerdo e piora quando faço esforço físico, hoje não consegui nem subir a escada de casa.
  Início dos sintomas: ha dois dias
  Sintomas identificados: braço esquerdo, dor no peito, dor no peito em aperto, esforço físico, piora quando faço esforço
  Diagnóstico sugerido: Infarto Agudo do Miocárdio (pontuação 5)
  Outras possibilidades: Miocardite (1), Pericardite (1)

Paciente 2
  Relato: Sinto cansaço constante há uma semana, mesmo depois de descansar, minhas pernas ficam inchadas no fim do dia e preciso dormir com dois travesseiros para conseguir respirar.
  Início dos sintomas: ha uma semana
  Sintomas identificados: cansaço, cansaço constante, dois travesseiros, inchadas no fim do dia, mesmo depois de descansar
  Diagnóstico sugerido: Insuficiência Cardíaca (pontuação 4)

Paciente 3
  Relato: Desde ontem sinto o coração disparado e umas palpitações do nada, fiquei tonto no trabalho e tive q

In [47]:
df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv("resultado_diagnosticos.csv", index=False, encoding="utf-8")
df_resultados[["paciente", "inicio_sintomas", "diagnostico_sugerido", "pontuacao", "outras_possibilidades"]]

,paciente,inicio_sintomas,diagnostico_sugerido,pontuacao,outras_possibilidades
0,1,ha dois dias,Infarto Agudo do Miocárdio,5,Miocardite (1); Pericardite (1)
1,2,ha uma semana,Insuficiência Cardíaca,4,
2,3,desde ontem,Arritmia,4,
3,4,ha tres semanas,Angina,7,Miocardite (1); Infarto Agudo do Miocárdio (1)
4,5,faz uns dez dias,Hipertensão Arterial,3,
5,6,hoje de manha,Acidente Vascular Cerebral (AVC),4,
6,7,ha quatro dias,Pericardite,4,Miocardite (1); Infarto Agudo do Miocárdio (1)
7,8,ha duas semanas,Miocardite,5,Arritmia (1); Infarto Agudo do Miocárdio (1)
8,9,ha cinco dias,Trombose Venosa Profunda / Embolia Pulmonar,6,Miocardite (1); Infarto Agudo do Miocárdio (1)
9,10,ha um mes,Endocardite Infecciosa,8,Pericardite (1)


### Teste de generalização: relatos que o mapa nunca viu
O mapa de conhecimento foi montado junto com as 10 frases, então acertar as 10 não prova que o sistema funciona com relatos novos. Aqui testamos 8 relatos escritos com outras palavras: variações de gênero, número e verbo ("tonta", "palpitação", "incharam"), negação e palavras intercaladas. Esses relatos **não** foram usados para ajustar o mapa.

In [49]:
relatos_novos = [
    ("Faz dois dias que sinto dores no peito e o braço esquerdo formiga quando faço esforço", "Infarto Agudo do Miocárdio"),
    ("Estou muito cansado há semanas, mesmo depois de descansar, e meus pés ficam inchados à noite", "Insuficiência Cardíaca"),
    ("Tenho tido tonturas e sinto o coração disparando do nada", "Arritmia"),
    ("Não sinto dor no peito, só dor de cabeça na nuca e a visão embaçada desde ontem", "Hipertensão Arterial"),
    ("Minha panturrilha inchou depois de uma viagem longa e agora sinto falta de ar de repente", "Trombose Venosa Profunda / Embolia Pulmonar"),
    ("Fico tonta e sinto uma palpitação estranha há três dias", "Arritmia"),
    ("Minhas pernas incharam muito e acordo sem ar no meio da noite", "Insuficiência Cardíaca"),
    ("Meu braço ficou dormente e a fala está arrastada desde hoje cedo", "Acidente Vascular Cerebral (AVC)"),
]

linhas = []
for relato, esperado in relatos_novos:
    ranking, sintomas = diagnosticar(relato)
    sugerido = ranking[0][0] if ranking else "Sem sugestão"
    linhas.append({
        "relato": relato,
        "esperado": esperado,
        "sugerido": sugerido,
        "acertou": sugerido == esperado,
        "sintomas_identificados": "; ".join(sintomas),
    })

df_teste = pd.DataFrame(linhas)
print(f"Acertos em relatos novos: {df_teste['acertou'].sum()} de {len(df_teste)}")
df_teste

Acertos em relatos novos: 7 de 8


,relato,esperado,sugerido,acertou,sintomas_identificados
0,Faz dois dias que sinto dores no peito e o bra...,Infarto Agudo do Miocárdio,Infarto Agudo do Miocárdio,True,braço esquerdo; dor no peito
1,"Estou muito cansado há semanas, mesmo depois d...",Insuficiência Cardíaca,Insuficiência Cardíaca,True,cansaço; mesmo depois de descansar
2,Tenho tido tonturas e sinto o coração disparan...,Arritmia,Arritmia,True,coração disparado; tonto
3,"Não sinto dor no peito, só dor de cabeça na nu...",Hipertensão Arterial,Hipertensão Arterial,True,dor de cabeça; dor de cabeça na nuca; nuca; vi...
4,Minha panturrilha inchou depois de uma viagem ...,Trombose Venosa Profunda / Embolia Pulmonar,Trombose Venosa Profunda / Embolia Pulmonar,True,falta de ar; falta de ar de repente; panturril...
5,Fico tonta e sinto uma palpitação estranha há ...,Arritmia,Arritmia,True,palpitações; tonto
6,Minhas pernas incharam muito e acordo sem ar n...,Insuficiência Cardíaca,Insuficiência Cardíaca,True,acordo sem ar; pernas inchadas
7,Meu braço ficou dormente e a fala está arrasta...,Acidente Vascular Cerebral (AVC),Sem sugestão,False,


### Observações e limitações

- **Generalização:** o mapa foi montado junto com as 10 frases, por isso acertar as 10 não mede generalização. Nos 8 relatos novos, o sistema acertou 7.
- **Palavras intercaladas:** a busca exige a expressão em sequência. "Braço ficou dormente" não casa com "braço dormente", e o relato de AVC ficou sem sugestão.
- **Negação simples:** olhamos até 3 palavras antes do sintoma. Uma frase como "a dor no peito não voltou" ainda conta o sintoma.
- **Sintomas genéricos:** "dor no peito", "febre" e "falta de ar" aparecem em várias doenças, por isso mostramos também as outras possibilidades, e não só uma resposta.
- **Origem do mapa:** o mapa foi montado pelo grupo a partir de material de estudo, não por um médico. O resultado é uma **sugestão de apoio**, nunca um diagnóstico final.
